# Final 03: Mixed Emotion End-to-End Orchestration and Paper-Ready Outputs

This notebook is the third step of the final workflow. It is an orchestration/evaluation notebook, not a one-click replacement for Final 01 and Final 02.

Run order:

1. Run Final 01 to create DistilBERT Phase 1 outputs.
2. Run Final 02 to create Llama 2 / Llama 3 Phase 2 outputs for routed rows.
3. Run this notebook to merge the files and generate paper-ready metrics, tables, and figures.

This notebook does not train DistilBERT and does not run Llama inference. It only reads saved CSV outputs from the previous two notebooks and constructs final end-to-end predictions.


In [ ]:
# Colab setup. This notebook only needs lightweight analysis packages.
# Do not upgrade numpy in-place; Colab's preinstalled numpy build is usually safest.
%pip install -q -U pandas scikit-learn matplotlib seaborn openpyxl
print("SETUP COMPLETE. If imports fail after package installation, restart the runtime once and rerun from the imports cell.")



## Imports and Persistent Output

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

LABELS = ["Depression", "Neutral", "Happy"]



In [ ]:
# Persistent output configuration.
# The notebook stops if Google Drive is unavailable. This prevents long runs from saving only to /content.
USE_GOOGLE_DRIVE_OUTPUT = True
REQUIRE_PERSISTENT_OUTPUT = True
LOCAL_OUTPUT_ROOT = Path("outputs_final")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final")

OUTPUT_ROOT = LOCAL_OUTPUT_ROOT
DRIVE_OUTPUT_AVAILABLE = False

if USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        probe_path = DRIVE_OUTPUT_ROOT / "_drive_write_test.txt"
        probe_path.write_text("ok", encoding="utf-8")
        probe_path.unlink(missing_ok=True)
        OUTPUT_ROOT = DRIVE_OUTPUT_ROOT
        DRIVE_OUTPUT_AVAILABLE = True
        print(f"Google Drive output enabled: {OUTPUT_ROOT}")
    except Exception as exc:
        if REQUIRE_PERSISTENT_OUTPUT:
            raise RuntimeError(
                "Google Drive output is unavailable, so the notebook stopped before running expensive work. "
                "Fix Drive authorization/mount first, or set REQUIRE_PERSISTENT_OUTPUT = False only for a temporary smoke test."
            ) from exc
        print(f"Google Drive output is unavailable ({exc}); falling back to local runtime output.")
        OUTPUT_ROOT = LOCAL_OUTPUT_ROOT
else:
    if REQUIRE_PERSISTENT_OUTPUT:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE_OUTPUT is False while REQUIRE_PERSISTENT_OUTPUT is True. "
            "Turn on Drive output or set REQUIRE_PERSISTENT_OUTPUT = False for a temporary run."
        )

END_TO_END_OUTPUT_DIR = OUTPUT_ROOT / "end_to_end_orchestration"
END_TO_END_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("End-to-end output directory:", END_TO_END_OUTPUT_DIR)


## Input Paths

Final 03 expects the following files to already exist in Google Drive:

- `outputs_final/phase1_distilbert/phase1_mixed_emotion_predictions.csv`
- `outputs_final/phase2_llm_reasoning/llama2_cot_routed_mixed_emotion_results.csv`
- `outputs_final/phase2_llm_reasoning/llama3_self_discover_routed_mixed_emotion_results.csv`

If only Llama 2 or only Llama 3 has finished, the notebook still runs and generates the available end-to-end comparison. For the final manuscript table, run both Phase 2 models first.


In [ ]:
PHASE1_PREDICTIONS_PATH = OUTPUT_ROOT / "phase1_distilbert" / "phase1_mixed_emotion_predictions.csv"
LLAMA2_RESULTS_PATH = OUTPUT_ROOT / "phase2_llm_reasoning" / "llama2_cot_routed_mixed_emotion_results.csv"
LLAMA3_RESULTS_PATH = OUTPUT_ROOT / "phase2_llm_reasoning" / "llama3_self_discover_routed_mixed_emotion_results.csv"

print("Phase 1:", PHASE1_PREDICTIONS_PATH)
print("Llama 2:", LLAMA2_RESULTS_PATH)
print("Llama 3:", LLAMA3_RESULTS_PATH)


## Load Outputs

In [ ]:
if not PHASE1_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(f"Missing Phase 1 predictions: {PHASE1_PREDICTIONS_PATH}")
phase1 = pd.read_csv(PHASE1_PREDICTIONS_PATH)
print("phase1", phase1.shape)
display(phase1.head())

llama2 = pd.read_csv(LLAMA2_RESULTS_PATH) if LLAMA2_RESULTS_PATH.exists() else pd.DataFrame()
llama3 = pd.read_csv(LLAMA3_RESULTS_PATH) if LLAMA3_RESULTS_PATH.exists() else pd.DataFrame()
print("llama2", llama2.shape)
print("llama3", llama3.shape)

required_phase1 = {"example_id", "text", "target_label", "phase1_label", "phase1_confidence", "phase1_accepted", "phase1_routed"}
missing = required_phase1 - set(phase1.columns)
if missing:
    raise ValueError(f"Phase 1 file missing columns: {missing}")


## Merge and Build End-to-End Predictions

In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    for label in LABELS:
        if text.lower() == label.lower():
            return label
    for label in LABELS:
        if label.lower() in text.lower():
            return label
    return np.nan

base = phase1.copy()
base["phase1_label"] = base["phase1_label"].map(normalize_label)
base["target_label"] = base["target_label"].map(normalize_label)
base["phase1_routed"] = base["phase1_routed"].astype(str).str.lower().isin(["true", "1", "yes"])
base["phase1_accepted"] = ~base["phase1_routed"]

if not llama2.empty:
    keep = ["example_id", "LLaMA2_final_label"]
    base = base.merge(llama2[keep].drop_duplicates("example_id", keep="last"), on="example_id", how="left")
    base["LLaMA2_final_label"] = base["LLaMA2_final_label"].map(normalize_label)
else:
    base["LLaMA2_final_label"] = np.nan

if not llama3.empty:
    keep = ["example_id", "LLaMA3_final_label"]
    base = base.merge(llama3[keep].drop_duplicates("example_id", keep="last"), on="example_id", how="left")
    base["LLaMA3_final_label"] = base["LLaMA3_final_label"].map(normalize_label)
else:
    base["LLaMA3_final_label"] = np.nan

base["final_label_llama2"] = np.where(base["phase1_routed"] & base["LLaMA2_final_label"].notna(), base["LLaMA2_final_label"], base["phase1_label"])
base["final_source_llama2"] = np.where(base["phase1_routed"] & base["LLaMA2_final_label"].notna(), "llama2_cot", "phase1")
base["final_label_llama3"] = np.where(base["phase1_routed"] & base["LLaMA3_final_label"].notna(), base["LLaMA3_final_label"], base["phase1_label"])
base["final_source_llama3"] = np.where(base["phase1_routed"] & base["LLaMA3_final_label"].notna(), "llama3_self_discover", "phase1")

base["is_correct_phase1"] = base["phase1_label"] == base["target_label"]
base["is_correct_llama2_e2e"] = base["final_label_llama2"] == base["target_label"]
base["is_correct_llama3_e2e"] = base["final_label_llama3"] == base["target_label"]

out_path = END_TO_END_OUTPUT_DIR / "mixed_emotion_end_to_end_results.csv"
base.to_csv(out_path, index=False)
print("Saved:", out_path)
display(base.head())


## Metrics and Correction Analysis

In [ ]:
def metrics_for(df, pred_col, name):
    valid = df.dropna(subset=["target_label", pred_col]).copy()
    if valid.empty:
        return {"model": name, "rows": 0, "accuracy": np.nan, "macro_precision": np.nan, "macro_recall": np.nan, "macro_f1": np.nan}
    p, r, f1, _ = precision_recall_fscore_support(valid["target_label"], valid[pred_col], labels=LABELS, average="macro", zero_division=0)
    return {
        "model": name,
        "rows": len(valid),
        "accuracy": accuracy_score(valid["target_label"], valid[pred_col]),
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1,
    }


def classification_report_frame(df, pred_col, name):
    valid = df.dropna(subset=["target_label", pred_col]).copy()
    if valid.empty:
        return pd.DataFrame()
    report = classification_report(valid["target_label"], valid[pred_col], labels=LABELS, output_dict=True, zero_division=0)
    rows = []
    for label, values in report.items():
        if isinstance(values, dict):
            rows.append({"model": name, "label": label, **values})
        else:
            rows.append({"model": name, "label": label, "score": values})
    return pd.DataFrame(rows)


def confusion_matrix_frame(df, pred_col, name):
    valid = df.dropna(subset=["target_label", pred_col]).copy()
    cm = confusion_matrix(valid["target_label"], valid[pred_col], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    cm_df.index.name = "target_label"
    cm_df.columns.name = "predicted_label"
    long_df = cm_df.reset_index().melt(id_vars="target_label", var_name="predicted_label", value_name="count")
    long_df.insert(0, "model", name)
    return cm_df, long_df

metric_specs = [("phase1_label", "Phase 1 DistilBERT only")]
if base["LLaMA2_final_label"].notna().any():
    metric_specs.append(("LLaMA2_final_label", "Llama 2 routed only"))
    metric_specs.append(("final_label_llama2", "End-to-end with Llama 2"))
if base["LLaMA3_final_label"].notna().any():
    metric_specs.append(("LLaMA3_final_label", "Llama 3 routed only"))
    metric_specs.append(("final_label_llama3", "End-to-end with Llama 3"))

metrics_df = pd.DataFrame([metrics_for(base[base["phase1_routed"]] if "routed only" in name else base, col, name) for col, name in metric_specs])
metrics_path = END_TO_END_OUTPUT_DIR / "end_to_end_metrics_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
display(metrics_df)

report_frames = []
cm_long_frames = []
for col, name in metric_specs:
    eval_base = base[base["phase1_routed"]] if "routed only" in name else base
    report = classification_report_frame(eval_base, col, name)
    if not report.empty:
        report_frames.append(report)
    _, cm_long = confusion_matrix_frame(eval_base, col, name)
    cm_long_frames.append(cm_long)

classification_reports_df = pd.concat(report_frames, ignore_index=True) if report_frames else pd.DataFrame()
classification_reports_df.to_csv(END_TO_END_OUTPUT_DIR / "end_to_end_classification_reports.csv", index=False)
confusion_matrices_long_df = pd.concat(cm_long_frames, ignore_index=True) if cm_long_frames else pd.DataFrame()
confusion_matrices_long_df.to_csv(END_TO_END_OUTPUT_DIR / "end_to_end_confusion_matrices_long.csv", index=False)

correction_rows = []
for model_name, final_col, phase2_col in [("llama2", "final_label_llama2", "LLaMA2_final_label"), ("llama3", "final_label_llama3", "LLaMA3_final_label")]:
    if phase2_col not in base or not base[phase2_col].notna().any():
        continue
    routed = base[base["phase1_routed"]].copy()
    corrected = (~routed["is_correct_phase1"]) & (routed[phase2_col] == routed["target_label"])
    introduced = (routed["is_correct_phase1"]) & (routed[phase2_col].notna()) & (routed[phase2_col] != routed["target_label"])
    remaining = (~routed["is_correct_phase1"]) & (routed[phase2_col] != routed["target_label"])
    correction_rows.append({
        "phase2_model": model_name,
        "routed_rows": len(routed),
        "phase1_errors_in_routed": int((~routed["is_correct_phase1"]).sum()),
        "corrected_errors": int(corrected.sum()),
        "introduced_errors": int(introduced.sum()),
        "remaining_routed_errors": int(remaining.sum()),
        "net_error_reduction": int(corrected.sum()) - int(introduced.sum()),
    })
correction_df = pd.DataFrame(correction_rows)
correction_path = END_TO_END_OUTPUT_DIR / "phase2_correction_analysis.csv"
correction_df.to_csv(correction_path, index=False)
display(correction_df)

routing_summary = pd.DataFrame([{
    "total_rows": len(base),
    "accepted_by_phase1": int(base["phase1_accepted"].sum()),
    "routed_to_phase2": int(base["phase1_routed"].sum()),
    "coverage": float(base["phase1_accepted"].mean()),
    "routing_rate": float(base["phase1_routed"].mean()),
    "phase1_accuracy_all": float((base["phase1_label"] == base["target_label"]).mean()),
    "phase1_accuracy_accepted": float((base.loc[base["phase1_accepted"], "phase1_label"] == base.loc[base["phase1_accepted"], "target_label"]).mean()) if base["phase1_accepted"].any() else np.nan,
    "phase1_accuracy_routed": float((base.loc[base["phase1_routed"], "phase1_label"] == base.loc[base["phase1_routed"], "target_label"]).mean()) if base["phase1_routed"].any() else np.nan,
    "routing_threshold": float(base["routing_threshold"].dropna().iloc[0]) if "routing_threshold" in base and base["routing_threshold"].notna().any() else np.nan,
    "temperature": float(base["temperature"].dropna().iloc[0]) if "temperature" in base and base["temperature"].notna().any() else np.nan,
}])
routing_summary.to_csv(END_TO_END_OUTPUT_DIR / "routing_coverage_table.csv", index=False)
display(routing_summary)

label_distribution_rows = []
for col, name in metric_specs:
    eval_base = base[base["phase1_routed"]] if "routed only" in name else base
    counts = eval_base[col].value_counts(dropna=False).rename_axis("label").reset_index(name="count")
    counts.insert(0, "model", name)
    label_distribution_rows.append(counts)
label_distribution_df = pd.concat(label_distribution_rows, ignore_index=True)
label_distribution_df.to_csv(END_TO_END_OUTPUT_DIR / "prediction_label_distributions.csv", index=False)



## Paper-Ready Figures and Tables

In [ ]:
def save_cm(df, pred_col, title, filename):
    valid = df.dropna(subset=["target_label", pred_col])
    cm = confusion_matrix(valid["target_label"], valid[pred_col], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    cm_df.index.name = "target_label"
    cm_df.columns.name = "predicted_label"
    csv_path = END_TO_END_OUTPUT_DIR / filename.replace(".png", ".csv")
    cm_df.to_csv(csv_path)

    plt.figure(figsize=(5.2, 4.2))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Target")
    plt.title(title)
    plt.tight_layout()
    path = END_TO_END_OUTPUT_DIR / filename
    plt.savefig(path, dpi=220)
    plt.show()
    return path, csv_path

figure_paths = []
path, csv_path = save_cm(base, "phase1_label", "Phase 1 DistilBERT", "confusion_matrix_phase1.png")
figure_paths.append(path)
if base["LLaMA2_final_label"].notna().any():
    path, csv_path = save_cm(base, "final_label_llama2", "End-to-End with Llama 2", "confusion_matrix_llama2_e2e.png")
    figure_paths.append(path)
if base["LLaMA3_final_label"].notna().any():
    path, csv_path = save_cm(base, "final_label_llama3", "End-to-End with Llama 3", "confusion_matrix_llama3_e2e.png")
    figure_paths.append(path)

with pd.ExcelWriter(END_TO_END_OUTPUT_DIR / "paper_ready_tables.xlsx") as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="metrics_summary")
    routing_summary.to_excel(writer, index=False, sheet_name="routing_coverage")
    correction_df.to_excel(writer, index=False, sheet_name="correction_analysis")
    classification_reports_df.to_excel(writer, index=False, sheet_name="classification_reports")
    confusion_matrices_long_df.to_excel(writer, index=False, sheet_name="confusion_matrices_long")
    label_distribution_df.to_excel(writer, index=False, sheet_name="label_distributions")
    pd.crosstab(base["target_label"], base["phase1_label"]).to_excel(writer, sheet_name="cm_phase1")
    if base["LLaMA2_final_label"].notna().any():
        pd.crosstab(base["target_label"], base["final_label_llama2"]).to_excel(writer, sheet_name="cm_llama2_e2e")
    if base["LLaMA3_final_label"].notna().any():
        pd.crosstab(base["target_label"], base["final_label_llama3"]).to_excel(writer, sheet_name="cm_llama3_e2e")
print("Saved paper-ready workbook:", END_TO_END_OUTPUT_DIR / "paper_ready_tables.xlsx")



## Final Paper Output Review

This section prints and displays the main paper-ready outputs in one place so the final run can be visually checked before the zip export.


In [ ]:
print("=" * 80)
print("FINAL MIXED EMOTION END-TO-END REVIEW")
print("Output directory:", END_TO_END_OUTPUT_DIR)
print("Rows:", len(base))
print("=" * 80)

print("\n[1] Metrics summary")
display(metrics_df)

print("\n[2] Routing / coverage summary")
display(routing_summary)

print("\n[3] Phase 2 correction analysis")
display(correction_df)

print("\n[4] Prediction label distributions")
display(label_distribution_df)

print("\n[5] Classification reports")
display(classification_reports_df)

print("\n[6] Confusion matrices as tables")
for col, name in metric_specs:
    eval_base = base[base["phase1_routed"]] if "routed only" in name else base
    cm_df, _ = confusion_matrix_frame(eval_base, col, name)
    print(f"\n{name}")
    display(cm_df)

print("\n[7] Confusion matrix figures")
for fig_path in figure_paths:
    if Path(fig_path).exists():
        print(fig_path)
        display(Image(filename=str(fig_path)))

print("\n[8] Example rows: Phase 1 errors")
phase1_errors = base[~base["is_correct_phase1"]].copy()
display(phase1_errors[["example_id", "target_label", "phase1_label", "phase1_confidence", "phase1_routed", "text"]].head(20))
phase1_errors.to_csv(END_TO_END_OUTPUT_DIR / "phase1_error_examples.csv", index=False)

if base["LLaMA2_final_label"].notna().any():
    print("\n[9] Example rows: Llama 2 end-to-end errors")
    llama2_errors = base[~base["is_correct_llama2_e2e"]].copy()
    display(llama2_errors[["example_id", "target_label", "phase1_label", "LLaMA2_final_label", "final_label_llama2", "final_source_llama2", "text"]].head(20))
    llama2_errors.to_csv(END_TO_END_OUTPUT_DIR / "llama2_e2e_error_examples.csv", index=False)

if base["LLaMA3_final_label"].notna().any():
    print("\n[10] Example rows: Llama 3 end-to-end errors")
    llama3_errors = base[~base["is_correct_llama3_e2e"]].copy()
    display(llama3_errors[["example_id", "target_label", "phase1_label", "LLaMA3_final_label", "final_label_llama3", "final_source_llama3", "text"]].head(20))
    llama3_errors.to_csv(END_TO_END_OUTPUT_DIR / "llama3_e2e_error_examples.csv", index=False)

print("\n[11] Paper-ready files")
for p in sorted(END_TO_END_OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"- {p.name} | {p.stat().st_size:,} bytes")



## Final Export

In [ ]:
OUTPUT_DIR_FOR_EXPORT = END_TO_END_OUTPUT_DIR
EXPORT_ZIP_NAME = "mixed_emotion_end_to_end_paper_outputs"
FINAL_MODEL_DIR = None
# Final export / download cell.
# This creates one zip file containing all available outputs from this notebook.
from pathlib import Path
import zipfile

files_to_zip = []
for pattern in ["*.csv", "*.json", "*.png", "*.xlsx"]:
    files_to_zip.extend(sorted(OUTPUT_DIR_FOR_EXPORT.rglob(pattern)))

# Include saved model files if present, but avoid adding huge checkpoint internals repeatedly.
model_dir = globals().get("FINAL_MODEL_DIR")
if model_dir is not None and Path(model_dir).exists():
    for p in Path(model_dir).glob("*"):
        if p.is_file():
            files_to_zip.append(p)

print("Files found for export:")
for p in files_to_zip:
    print(f"- {p} | {p.stat().st_size:,} bytes")

if not files_to_zip:
    print("No output files found yet.")
else:
    zip_path = OUTPUT_DIR_FOR_EXPORT / f"{EXPORT_ZIP_NAME}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in files_to_zip:
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR_FOR_EXPORT) if p.is_relative_to(OUTPUT_DIR_FOR_EXPORT) else p.name)
    print(f"Saved zip: {zip_path} | {zip_path.stat().st_size:,} bytes")
    if DRIVE_OUTPUT_AVAILABLE:
        print("Persistent Drive copy is available here:")
        print(zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Automatic browser download was not started: {exc}")

